# FAR tests using different datasets

In [1]:
import numpy as np
import pandas as pd
import sklearn as skl
import zipfile as zf
import matplotlib.pyplot as plt
import nbimporter
import shap

import Ft_Att_Rank as far

# Iris dataset

In [2]:
import sklearn.datasets

iris= sklearn.datasets.load_iris()

X_iris= iris.data
Y_iris= iris.target

In [3]:
# convert iris to a df and drop the setosa rows
df_iris= pd.DataFrame(data= np.c_[X_iris, Y_iris], columns= iris['feature_names'] + ['target'])
df_iris= df_iris[df_iris['target']!= 0].reset_index(drop= True)

In [4]:
# split df_iris into features (x) and target (y)
df_iris_x= df_iris.loc[:,df_iris.columns[0:4]]
df_iris_y= df_iris.loc[:,df_iris.columns[4:5]]

df_iris_x= far.normalize_selected_cols(df_iris_x,df_iris_x.columns)

df_iris_x.shape

(100, 4)

In [5]:
# ML model - Random forest
import sklearn.ensemble

train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(df_iris_x,df_iris_y,train_size=0.80,random_state=1234)

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)
rf.fit(train, labels_train.values.ravel())
rf_acc= sklearn.metrics.accuracy_score(labels_test, rf.predict(test))

rf_acc

0.85

In [7]:
# ML model - XGBoost random forest
import xgboost as xgb

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',verbosity=0)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.85

In [9]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train.values.ravel(), labels_test.values.ravel(), repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [10]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

# get the stationary distribution
stationary_d1= far.stationary_dist_v1(st_matrix1)
stationary_d2= far.stationary_dist_v1(st_matrix2)
stationary_d3= far.stationary_dist_v1(st_matrix3)
stationary_d4= far.stationary_dist_v1(st_matrix4)

4.0
3.0
3.0
4.0


In [11]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.459770
petal width (cm)     0.402299
sepal length (cm)    0.137931
sepal width (cm)     0.000000
dtype: float64

In [12]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.389768
petal width (cm)     0.300783
sepal length (cm)    0.174804
sepal width (cm)     0.129429
dtype: float64

In [13]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.353756
petal width (cm)     0.287302
sepal length (cm)    0.197763
sepal width (cm)     0.153875
dtype: float64

In [14]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.460100
petal width (cm)     0.427681
sepal length (cm)    0.112219
sepal width (cm)     0.000000
dtype: float64

# Titanic dataset

In [15]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [16]:
X_train= far.pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')
X_train= far.pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train= far.normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= far.normalize_selected_cols(X_train_ohe, numeric_columns)

X_train.shape

(891, 7)

In [17]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.8379888268156425

In [18]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,numeric_columns,num_type='mean',cat_type='median')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [19]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

# get the stationary distribution
stationary_d1= far.stationary_dist_v1(st_matrix1)
stationary_d2= far.stationary_dist_v1(st_matrix2)
stationary_d3= far.stationary_dist_v1(st_matrix3)
stationary_d4= far.stationary_dist_v1(st_matrix4)

12.0
11.0
11.0
12.0


In [20]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

Age           2.609210e-01
Pclass_3      1.195357e-01
Sex_male      1.148095e-01
Sex_female    1.147950e-01
Embarked_S    9.569253e-02
Fare          6.552390e-02
Parch         6.205284e-02
SibSp         4.868189e-02
Pclass_2      4.267618e-02
Embarked_Q    3.927520e-02
Pclass_1      3.603621e-02
Embarked_C   -6.212792e-17
dtype: float64

In [21]:
far.ft_importance_df(stationary_d1,train.columns)

,Feature,Importance
0,Age,0.260921
1,Sex,0.229605
2,Pclass,0.198248
3,Embarked,0.134968
4,Fare,0.065524
5,Parch,0.062053
6,SibSp,0.048682


In [22]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.253968
Sex_female    0.247595
Age           0.099870
Pclass_3      0.066443
Fare          0.066196
SibSp         0.047784
Embarked_S    0.047293
Pclass_1      0.037691
Pclass_2      0.037207
Parch         0.035730
Embarked_Q    0.034735
Embarked_C    0.025430
dtype: float64

In [23]:
far.ft_importance_df(stationary_d2,train.columns)

,Feature,Importance
0,Sex,0.501563
1,Pclass,0.141340
2,Embarked,0.107458
3,Age,0.099870
4,Fare,0.066196
5,SibSp,0.047784
6,Parch,0.035730


In [24]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

Sex_male      0.270839
Sex_female    0.265461
Age           0.091642
Fare          0.067300
Pclass_3      0.056732
Embarked_S    0.048883
Parch         0.037776
SibSp         0.035983
Pclass_2      0.035773
Pclass_1      0.033484
Embarked_Q    0.032184
Embarked_C    0.023891
dtype: float64

In [25]:
far.ft_importance_df(stationary_d2,train.columns)

,Feature,Importance
0,Sex,0.501563
1,Pclass,0.141340
2,Embarked,0.107458
3,Age,0.099870
4,Fare,0.066196
5,SibSp,0.047784
6,Parch,0.035730


In [26]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

Age           3.104739e-01
Embarked_S    1.415517e-01
Pclass_3      1.282012e-01
Fare          1.110406e-01
Sex_female    6.062750e-02
Sex_male      5.800315e-02
Parch         5.494874e-02
SibSp         4.060065e-02
Pclass_2      3.696395e-02
Embarked_Q    2.974330e-02
Pclass_1      2.784524e-02
Embarked_C    1.574925e-17
dtype: float64

In [27]:
far.ft_importance_df(stationary_d4,train.columns)

,Feature,Importance
0,Age,0.310474
1,Pclass,0.193010
2,Embarked,0.171295
3,Sex,0.118631
4,Fare,0.111041
5,Parch,0.054949
6,SibSp,0.040601


# Heartrisk dataset

In [28]:
ds= zf.ZipFile('datasets/heart.zip')

heart_data= pd.read_csv(ds.open('heart.csv'))

x_heart= heart_data.iloc[:,:(len(heart_data.columns)-1)].copy()
y_heart= np.asarray(heart_data['target'])

x_heart.shape

(303, 13)

In [29]:
x_heart= far.pre_proc_fillna_num_fts(x_heart, x_heart.columns,num_type='mean')

x_heart= far.normalize_selected_cols(x_heart, x_heart.columns)

In [30]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_heart,y_heart,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.7377049180327869

In [31]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [32]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

# get the stationary distribution
stationary_d1= far.stationary_dist_v1(st_matrix1)
stationary_d2= far.stationary_dist_v1(st_matrix2)
stationary_d3= far.stationary_dist_v1(st_matrix3)
stationary_d4= far.stationary_dist_v1(st_matrix4)

13.0
11.999999999999998
12.0
13.0


In [33]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          1.599029e-01
ca          1.559408e-01
thalach     1.489579e-01
oldpeak     9.772164e-02
slope       8.161149e-02
thal        8.052835e-02
sex         7.767256e-02
exang       7.291755e-02
chol        4.474518e-02
trestbps    2.771242e-02
restecg     2.732068e-02
age         2.496851e-02
fbs        -7.071517e-18
dtype: float64

In [34]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          0.175510
oldpeak     0.111838
ca          0.099591
restecg     0.077959
slope       0.073683
fbs         0.073213
thalach     0.070996
trestbps    0.067910
exang       0.052447
chol        0.051907
age         0.050840
thal        0.048261
sex         0.045403
dtype: float64

In [35]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          0.172361
oldpeak     0.113941
ca          0.088451
slope       0.081183
trestbps    0.065897
restecg     0.065284
chol        0.065270
thalach     0.065071
thal        0.061880
fbs         0.059755
sex         0.056819
age         0.053116
exang       0.050676
dtype: float64

In [36]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          2.349326e-01
thalach     1.469698e-01
ca          1.244601e-01
thal        9.662509e-02
oldpeak     9.370323e-02
exang       7.505775e-02
chol        7.085382e-02
slope       5.311463e-02
sex         5.171700e-02
age         1.871688e-02
trestbps    1.708402e-02
restecg     1.676500e-02
fbs         1.293062e-17
dtype: float64

# Wine dataset

In [37]:
wine= pd.read_csv('datasets/wine.data',header=None)

wine.columns= ['target','alcohol','malicAcid','ash','ashalcalinity','magnesium','totalPhenols','flavanoids','nonFlavanoidPhenols','proanthocyanins',
               'colorIntensity','hue','od280_od315','proline']

wine= wine[wine['target']!= 3].reset_index(drop= True)

x_wine= wine.iloc[:,1:len(wine.columns)].copy()
y_wine= np.asarray(wine['target'])

x_wine.shape

(130, 13)

In [38]:
x_wine= far.pre_proc_fillna_num_fts(x_wine, x_wine.columns, num_type='mean')

x_wine= far.normalize_selected_cols(x_wine, x_wine.columns)

In [40]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_wine,y_wine,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss')
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

1.0

In [41]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [42]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

# get the stationary distribution
stationary_d1= far.stationary_dist_v1(st_matrix1)
stationary_d2= far.stationary_dist_v1(st_matrix2)
stationary_d3= far.stationary_dist_v1(st_matrix3)
stationary_d4= far.stationary_dist_v1(st_matrix4)

13.0
5.0
5.0
13.0


In [43]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         3.750000e-01
flavanoids             2.031250e-01
alcohol                2.031250e-01
proline                1.250000e-01
malicAcid              9.375000e-02
proanthocyanins        6.161092e-17
ash                    5.998215e-17
magnesium              5.897334e-17
nonFlavanoidPhenols    5.285098e-17
hue                    4.645137e-17
od280_od315            4.619216e-17
totalPhenols           1.158999e-17
ashalcalinity          3.574815e-18
dtype: float64

In [44]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         0.224134
proline                0.124110
alcohol                0.120550
flavanoids             0.120550
malicAcid              0.092353
totalPhenols           0.038406
nonFlavanoidPhenols    0.038406
magnesium              0.038406
proanthocyanins        0.038406
ash                    0.038406
hue                    0.038406
od280_od315            0.038406
ashalcalinity          0.038406
dtype: float64

In [45]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         0.215610
proline                0.129927
alcohol                0.122626
flavanoids             0.122626
malicAcid              0.084595
magnesium              0.039155
totalPhenols           0.039155
ashalcalinity          0.039155
proanthocyanins        0.039155
nonFlavanoidPhenols    0.039155
hue                    0.039155
od280_od315            0.039155
ash                    0.039155
dtype: float64

In [46]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         3.600823e-01
proline                2.386831e-01
alcohol                1.646091e-01
flavanoids             1.646091e-01
malicAcid              7.201646e-02
magnesium              6.225208e-17
ashalcalinity          4.847039e-17
ash                    4.591939e-17
totalPhenols           2.923372e-17
od280_od315            1.441089e-17
hue                    1.430162e-17
nonFlavanoidPhenols   -1.081462e-17
proanthocyanins       -1.243046e-17
dtype: float64

# Breast cancer Wisconsin dataset

In [47]:
wdbc= sklearn.datasets.load_breast_cancer()

X_wb_cancer= pd.DataFrame(wdbc.data,columns=wdbc.feature_names)
Y_wb_cancer= wdbc.target

X_wb_cancer.shape

(569, 30)

In [48]:
X_wb_cancer= far.pre_proc_fillna_num_fts(X_wb_cancer,wdbc.feature_names,num_type='mean')

X_wb_cancer= far.normalize_selected_cols(X_wb_cancer, wdbc.feature_names)

In [49]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_wb_cancer,Y_wb_cancer,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.9298245614035088

In [50]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [51]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= far.to_row_stochastic_matrix(st_matrix1)
st_matrix2= far.to_row_stochastic_matrix(st_matrix2)
st_matrix3= far.to_row_stochastic_matrix(st_matrix3)
st_matrix4= far.to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

# get the stationary distribution
stationary_d1= far.stationary_dist_v1(st_matrix1)
stationary_d2= far.stationary_dist_v1(st_matrix2)
stationary_d3= far.stationary_dist_v1(st_matrix3)
stationary_d4= far.stationary_dist_v1(st_matrix4)

30.0
30.0
30.0
30.0


In [52]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              0.126582
mean texture               0.125821
worst perimeter            0.082647
worst concave points       0.082275
mean concave points        0.072930
area error                 0.064593
worst area                 0.061003
worst smoothness           0.054675
mean concavity             0.052254
worst radius               0.046184
worst concavity            0.040249
mean radius                0.031334
mean area                  0.027534
mean smoothness            0.021390
worst compactness          0.018487
mean perimeter             0.014755
symmetry error             0.013531
worst fractal dimension    0.011882
worst symmetry             0.008634
perimeter error            0.008088
concavity error            0.007906
mean compactness           0.007197
smoothness error           0.005960
fractal dimension error    0.004486
compactness error          0.003167
texture error              0.002145
concave points error       0.001253
radius error               0

In [53]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

mean radius                0.139078
mean smoothness            0.089419
worst concavity            0.076192
worst radius               0.053053
mean area                  0.051504
fractal dimension error    0.039974
compactness error          0.035179
worst compactness          0.034068
mean concavity             0.031654
radius error               0.030051
concave points error       0.030051
worst symmetry             0.029933
mean perimeter             0.027936
worst texture              0.027124
concavity error            0.026980
worst concave points       0.026484
mean compactness           0.025700
mean texture               0.025443
worst area                 0.024769
texture error              0.023664
worst fractal dimension    0.023188
perimeter error            0.019313
mean concave points        0.016994
worst smoothness           0.016779
mean symmetry              0.015483
mean fractal dimension     0.015483
symmetry error             0.013688
smoothness error           0

In [54]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

mean radius                0.129725
mean smoothness            0.091818
worst concavity            0.070572
worst radius               0.053177
mean area                  0.053077
fractal dimension error    0.037799
mean concavity             0.033859
worst texture              0.031030
compactness error          0.030433
worst concave points       0.030198
mean perimeter             0.029310
worst compactness          0.028199
worst symmetry             0.027408
concavity error            0.026479
mean compactness           0.025906
radius error               0.025290
concave points error       0.025290
worst area                 0.025150
mean texture               0.024361
perimeter error            0.023771
texture error              0.023155
worst fractal dimension    0.022885
mean concave points        0.021069
worst smoothness           0.021032
worst perimeter            0.016732
mean symmetry              0.015524
mean fractal dimension     0.015524
symmetry error             0

In [55]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              0.124921
worst concave points       0.098545
worst perimeter            0.098536
mean texture               0.095928
worst smoothness           0.086161
mean concave points        0.068941
area error                 0.065571
mean concavity             0.057345
worst concavity            0.049666
mean area                  0.047056
mean smoothness            0.044621
worst compactness          0.039751
worst area                 0.031141
worst radius               0.025735
mean radius                0.013694
mean perimeter             0.007909
symmetry error             0.007509
worst fractal dimension    0.006105
worst symmetry             0.004919
perimeter error            0.004908
concavity error            0.004592
smoothness error           0.004363
mean compactness           0.003764
fractal dimension error    0.002986
texture error              0.001478
mean fractal dimension     0.001144
mean symmetry              0.001144
compactness error          0